# Notebook 2: Train Continual SD-LoRA Adapter

Bu notebook tek crop icin adapter egitir, OOD hazirligini olcer ve export ciktisini kaydeder.

Onerilen kullanim sirasi:
1. Calisma kimligi hucresinde `CROP_NAME` ve `PART_NAME` belirleyin.
2. Parametreleri duzenleyin.
3. Guncelleme/erisim kontrolu hucresini calistirin.
4. Hucreleri sirayla calistirin.
5. Is bitince `guided/00_start_here.md` ile ciktilari okuyun.


In [1]:
from pathlib import Path
import os
import shutil
import subprocess
import sys

CLONE_TARGET = Path('/content/bitirmeprojesi')
REPO_URL = os.environ.get('AADS_REPO_URL', 'https://github.com/EfeErim/bitirmeprojesi.git')
SPARSE_PATHS = [
    'README.md', 'docs', 'src', 'scripts', 'config', 'colab_notebooks', 'requirements.txt', 'requirements_colab.txt', 'pyproject.toml',
]
REQUIRED_BOOTSTRAP_FILES = ['requirements_colab.txt']

def _repair_sparse_checkout(repo_root):
    missing = [path for path in REQUIRED_BOOTSTRAP_FILES if not (repo_root / path).exists()]
    if missing and (repo_root / '.git').exists():
        print(f'[BOOTSTRAP] Adding missing sparse checkout paths: {missing}', flush=True)
        subprocess.run(['git', 'sparse-checkout', 'add', *missing], cwd=str(repo_root), check=True)

def _ensure_aads_repo_on_path():
    candidates = [Path.cwd(), Path.cwd().parent, CLONE_TARGET, Path('/content/bitirmeprojesi'), Path('/content/bitirme projesi')]
    for candidate in candidates:
        marker = candidate / 'scripts' / 'notebook_helpers' / 'cell_script_runner.py'
        if marker.is_file():
            repo_root = candidate.resolve()
            _repair_sparse_checkout(repo_root)
            if str(repo_root) not in sys.path:
                sys.path.insert(0, str(repo_root))
            return repo_root
    if CLONE_TARGET.exists():
        shutil.rmtree(CLONE_TARGET)
    print(f'[BOOTSTRAP] Sparse cloning training surface to {CLONE_TARGET}...', flush=True)
    subprocess.run(['git', 'clone', '--depth', '1', '--filter=blob:none', '--sparse', REPO_URL, str(CLONE_TARGET)], check=True)
    print('[BOOTSTRAP] Selecting source checkout paths...', flush=True)
    subprocess.run(['git', 'sparse-checkout', 'set', *SPARSE_PATHS], cwd=str(CLONE_TARGET), check=True)
    if str(CLONE_TARGET) not in sys.path:
        sys.path.insert(0, str(CLONE_TARGET))
    return CLONE_TARGET

_ensure_aads_repo_on_path()

from scripts.notebook_helpers.cell_script_runner import run_cell_script
run_cell_script('nb2_cell01_bootstrap_access.py', globals())


[BOOTSTRAP] Sparse cloning training surface to /content/bitirmeprojesi...
[BOOTSTRAP] Selecting source checkout paths...
[BOOTSTRAP] GitHub token found.
[BOOTSTRAP] HuggingFace token found.
[BOOTSTRAP] Repo root: /content/bitirmeprojesi
[BOOTSTRAP] Installing requirements_colab.txt...
[BOOTSTRAP] Notebook 2: Continual Adapter Training bootstrap complete.

BOOTSTRAP STATUS
Status: ok
Repo Root: /content/bitirmeprojesi
In Colab: True
GitHub Token: ✓
HuggingFace Token: ✓

[SETUP] Checking model access for training...
[KONTROL] Repo guncel gorunuyor.
[KONTROL] GitHub okuma erisimi public; clone/pull icin ekstra token gerekmiyor.
[KONTROL] GitHub push icin kimlik bilgisi hazir.
[KONTROL] Gerekli Hugging Face modelleri anonim erisimle aciliyor.
[KONTROL] Varsayilan backbone modeli: facebook/dinov3-vitl16-pretrain-lvd1689m


In [2]:
# Notebook 2 calisma kimligi
# Once bu hucreyi duzenleyin, sonra bootstrap hucresini yeniden calistirin.
CROP_NAME = "strawberry"
PART_NAME = "leaf"
ENABLE_BAYESIAN_OPTIMIZATION = False  # Bayesian optimization devre disi
print(f"[RUN] crop={CROP_NAME} part={PART_NAME} bayes_opt={ENABLE_BAYESIAN_OPTIMIZATION}")


[RUN] crop=strawberry part=leaf bayes_opt=False


In [3]:
from scripts.notebook_helpers.cell_script_runner import run_cell_script
run_cell_script('nb2_cell03_runtime_setup.py', globals())

[BOOTSTRAP] run_id=strawberry_leaf_2026-05-04_20-57-37 crop=strawberry part=leaf


In [4]:
with TELEMETRY.capture_cell_output("Cell 3: Parameters"):
    # --- Notebook 2 parametreleri (strawberry__leaf, genel en iyi denge) ---
    RUNTIME_DATASET_ROOT = "data/prepared_runtime_datasets"
    DATASET_NAME = "strawberry__leaf"

    OOD_ROOT = "data/prepared_runtime_datasets/strawberry__leaf/ood"
    ASK_FOR_OOD_ROOT = False

    OE_ROOT = "data/prepared_runtime_datasets/strawberry__leaf/oe"
    ASK_FOR_OE_ROOT = False
    OE_ENABLED = True
    OE_LOSS_WEIGHT = 0.35

    CROP_NAME = globals().get("CROP_NAME", "strawberry")
    PART_NAME = globals().get("PART_NAME", "leaf")
    ENABLE_BAYESIAN_OPTIMIZATION = bool(globals().get("ENABLE_BAYESIAN_OPTIMIZATION", False))

    ALLOW_UNDER_MIN_TRAINING = False
    EPOCHS = 12
    BATCH_SIZE = 96
    LEARNING_RATE = 2e-4
    LORA_R = 24
    LORA_ALPHA = 24
    LORA_DROPOUT = 0.1

    OOD_FACTOR = 3.0
    SURE_SEMANTIC_PERCENTILE = 90.0
    SURE_CONFIDENCE_PERCENTILE = 97.0
    CONFORMAL_ALPHA = 0.05
    CONFORMAL_METHOD = "raps"
    CONFORMAL_RAPS_LAMBDA = 0.2
    CONFORMAL_RAPS_K_REG = 1

    BER_ENABLED = False
    BER_LAMBDA_OLD = 0.1
    BER_LAMBDA_NEW = 0.1
    BER_WARMUP_STEPS = 50

    WEIGHT_DECAY = 0.01
    MIXED_PRECISION = "bf16"
    GRAD_ACCUM_STEPS = 1
    MAX_GRAD_NORM = 1.0
    LABEL_SMOOTHING = 0.0

    LOSS_NAME = "logitnorm"
    LOGITNORM_TAU = 1.0

    SCHEDULER_NAME = "cosine"
    SCHEDULER_WARMUP_RATIO = 0.1
    SCHEDULER_MIN_LR = 1e-6

    EARLY_STOPPING_PATIENCE = 5
    EARLY_STOPPING_MIN_DELTA = 0.0

    DETERMINISTIC = False
    TF32_ENABLED = True
    SEED = 42

    NUM_WORKERS = 12
    PREFETCH = 8
    PIN_MEMORY = True
    USE_CACHE = True
    CACHE_TRAIN_SPLIT = True

    VALIDATION_EVERY_N_EPOCHS = 2
    CHECKPOINT_EVERY_N_STEPS = 250
    CHECKPOINT_ON_EXCEPTION = True
    STDOUT_BATCH_INTERVAL = 12

    RESUME_MODE = "fresh"
    AUTO_DISCONNECT_RUNTIME = True
    AUTO_DISCONNECT_GRACE_SECONDS = 20
    AUTO_PUSH_TO_GITHUB = True
    AUTO_PUSH_REMOTE_NAME = "origin"
    AUTO_PUSH_BRANCH = None

    MANUAL_PARAM_OVERRIDES = {}

    from scripts.notebook_helpers.cell_script_runner import run_cell_script
    run_cell_script('nb2_cell04_parameter_resolution.py', globals())

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


[HF] Kimlik dogrulandi: Grizzmann
[GIT] Auto-push on kontrolu gecti: GitHub token cozuldu.
[CONFIG] source=notebook_cell crop=strawberry epochs=12 bs=96 lr=0.0002 resume=fresh ber=False loss=logitnorm tau=1.0 auto_disconnect=True auto_push=True bayes_opt=False
[DATASET] runtime_root=data/prepared_runtime_datasets dataset_name=strawberry__leaf
[OOD] ood_root=data/prepared_runtime_datasets/strawberry__leaf/ood ask_for_ood_root=False
[OE] oe_root=data/prepared_runtime_datasets/strawberry__leaf/oe ask_for_oe_root=False enabled=True loss_weight=0.35
[RUNTIME] defaults=notebook_cell mp=bf16 workers=12 prefetch=8 sched=cosine wd=0.01 accum=1 grad_clip=1.0 label_smooth=0.0 warmup=0.1 early_stop=5/0.0 val_every=2 cache_train=True aug=randaugment randaug=2/7 
[OOD] factor=3.0 sure=90.0/97.0 conformal=raps alpha=0.05 raps_lambda=0.2 raps_k=1


In [5]:
from scripts.notebook_helpers.cell_script_runner import run_cell_script
run_cell_script('nb2_cell05_access_check.py', globals())


[KONTROL] Repo guncel gorunuyor.
[KONTROL] GitHub okuma erisimi public; clone/pull icin ekstra token gerekmiyor.
[KONTROL] GitHub push icin kimlik bilgisi hazir.
[KONTROL] Gerekli Hugging Face modelleri anonim erisimle aciliyor.


In [6]:
from scripts.notebook_helpers.cell_script_runner import run_cell_script
run_cell_script('nb2_cell06_dataset_validation.py', globals())


[BOOTSTRAP] Fetching selected Notebook 2 dataset paths: ['data/prepared_runtime_datasets/strawberry__leaf', 'data/ood_dataset/final/strawberry__leaf_ood_final', 'data/oe_dataset/strawberry_leaf_oe_from_blossom_candidates', 'data/prepared_runtime_datasets/strawberry__leaf/ood', 'data/prepared_runtime_datasets/strawberry__leaf/oe']
[OOD] explicit ood root=/content/bitirmeprojesi/data/prepared_runtime_datasets/strawberry__leaf/ood
[OE] explicit oe root=/content/bitirmeprojesi/data/prepared_runtime_datasets/strawberry__leaf/oe
[DATASET] runtime root=/content/bitirmeprojesi/data/prepared_runtime_datasets/strawberry__leaf classes=4: ['healthy', 'strawberry_leaf_scorch_leaf', 'strawberry_leaf_spot_leaf', 'strawberry_powdery_mildew_leaf']
[DATASET][CHECK] scale=small continual=1606 val=491 test=491 ood=292 classes=4
[PARAMS][FINAL] epochs=12 bs=96 lr=0.0002 lora=24/24 dropout=0.1 accum=1 workers=12 prefetch=8


In [7]:
from scripts.notebook_helpers.cell_script_runner import run_cell_script
run_cell_script('nb2_cell07_engine_init.py', globals())


[OOD] selected ood root=/content/bitirmeprojesi/data/prepared_runtime_datasets/strawberry__leaf/ood
[OE] selected oe root=/content/bitirmeprojesi/data/prepared_runtime_datasets/strawberry__leaf/oe
[CLASSES] ['healthy', 'strawberry_leaf_scorch_leaf', 'strawberry_leaf_spot_leaf', 'strawberry_powdery_mildew_leaf']
[CLASSES] mode=dataset_fallback reason=partial_taxonomy_alignment_fallback matched=1 unmatched=3
[CLASSES] taxonomy-unmatched classes kept: ['strawberry_leaf_scorch_leaf', 'strawberry_leaf_spot_leaf', 'strawberry_powdery_mildew_leaf']
[ENGINE][OOD_CFG] {"ber_enabled": false, "ber_lambda_new": 0.1, "ber_lambda_old": 0.1, "ber_warmup_steps": 50, "conformal_alpha": 0.05, "conformal_enabled": true, "conformal_method": "raps", "conformal_raps_k_reg": 1, "conformal_raps_lambda": 0.2, "energy_temperature": 1.0, "energy_temperature_mode": "auto", "energy_temperature_range": [0.5, 3.0], "energy_temperature_steps": 16, "knn_backend": "auto", "knn_chunk_size": 2048, "oe_enabled": true, "oe

config.json:   0%|          | 0.00/745 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.21G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/415 [00:00<?, ?it/s]

[ENGINE] Hazir. trainable_params=13,770,248  classes=4


In [8]:
from scripts.notebook_helpers.cell_script_runner import run_cell_script
run_cell_script('nb2_cell08_ood_config_verify.py', globals())


[VERIFY][OOD][expected] {"conformal_alpha": 0.05, "conformal_method": "raps", "conformal_raps_k_reg": 1, "conformal_raps_lambda": 0.2, "sure_confidence_percentile": 97.0, "sure_semantic_percentile": 90.0, "threshold_factor": 3.0}
[VERIFY][OOD][resolved] {"conformal_alpha": 0.05, "conformal_method": "raps", "conformal_raps_k_reg": 1, "conformal_raps_lambda": 0.2, "sure_confidence_percentile": 97.0, "sure_semantic_percentile": 90.0, "threshold_factor": 3.0}
[VERIFY][OOD] OK: cozulmus OOD ayari istenen parametrelerle eslesiyor.


In [9]:
from scripts.notebook_helpers.cell_script_runner import run_cell_script
run_cell_script('nb2_cell09_training.py', globals())


[TRAIN] epochs=12 device=cuda batch_interval=12
[TRAIN] epoch=1 batch=12/16 loss=0.0000 lr=0.000198 speed=266.1s/s elapsed=8s eta=124s


/content/bitirmeprojesi/scripts/notebook_cells/nb2_cell09_training.py:193: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
  plt.legend()
/content/bitirmeprojesi/scripts/notebook_cells/nb2_cell09_training.py:203: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
  plt.legend()


[EPOCH] 1/12: train_loss=1.4024
[TRAIN] epoch=2 batch=12/16 loss=0.0000 lr=0.000190 speed=266.1s/s elapsed=16s eta=95s
[CKPT] epoch_end epoch=2 step=32
[EPOCH] 2/12: train_loss=1.2589 val_loss=0.6218 val_acc=0.7352 macro_f1=0.6499 * BEST
[TRAIN] epoch=3 batch=12/16 loss=0.0000 lr=0.000175 speed=266.2s/s elapsed=29s eta=98s
[EPOCH] 3/12: train_loss=0.9309
[TRAIN] epoch=4 batch=12/16 loss=0.0000 lr=0.000156 speed=266.0s/s elapsed=36s eta=80s
[CKPT] epoch_end epoch=4 step=64
[EPOCH] 4/12: train_loss=0.7914 val_loss=0.1379 val_acc=0.9470 macro_f1=0.9448 * BEST
[TRAIN] epoch=5 batch=12/16 loss=0.0000 lr=0.000132 speed=266.5s/s elapsed=48s eta=74s
[EPOCH] 5/12: train_loss=0.7082
[TRAIN] epoch=6 batch=12/16 loss=0.0000 lr=0.000107 speed=266.0s/s elapsed=56s eta=61s
[CKPT] epoch_end epoch=6 step=96
[EPOCH] 6/12: train_loss=0.6862 val_loss=0.0203 val_acc=0.9959 macro_f1=0.9951 * BEST
[TRAIN] epoch=7 batch=12/16 loss=0.0000 lr=0.000081 speed=266.4s/s elapsed=68s eta=53s
[EPOCH] 7/12: train_loss=

In [10]:
from scripts.notebook_helpers.cell_script_runner import run_cell_script
run_cell_script('nb2_cell10_ood_calibration.py', globals())


[OOD] Kalibrasyon tamamlandi. classes=4 version=1


In [11]:
from scripts.notebook_helpers.cell_script_runner import run_cell_script
run_cell_script('nb2_cell11_adapter_save.py', globals())


Cell 8: adapter save started
Kaydedilen adapter klasoru: /content/bitirmeprojesi/outputs/colab_notebook_training/strawberry/leaf/continual_sd_lora_adapter
 - outputs/colab_notebook_training/strawberry/leaf/continual_sd_lora_adapter/README.md
 - outputs/colab_notebook_training/strawberry/leaf/continual_sd_lora_adapter/adapter_config.json
 - outputs/colab_notebook_training/strawberry/leaf/continual_sd_lora_adapter/adapter_meta.json
 - outputs/colab_notebook_training/strawberry/leaf/continual_sd_lora_adapter/adapter_model.safetensors
 - outputs/colab_notebook_training/strawberry/leaf/continual_sd_lora_adapter/classifier.pth
 - outputs/colab_notebook_training/strawberry/leaf/continual_sd_lora_adapter/fusion.pth
Telemetry adapter klasoru: /content/bitirmeprojesi/outputs/colab_notebook_training/telemetry_runtime/telemetry/strawberry_leaf_2026-05-04_20-57-37/artifacts/adapter_export/strawberry/leaf/continual_sd_lora_adapter
Cell 8: adapter save completed


In [ ]:
from scripts.notebook_helpers.cell_script_runner import run_cell_script
run_cell_script('nb2_cell12_final_evaluation.py', globals())


[DOGRULAMA (referans)] ornek=491 sinif=4 accuracy=0.9939 ood_auroc=0.9334 sure_ds_f1=0.7305 conformal_cov=0.9939
[TEST (ayrilmis)] ornek=491 sinif=4 accuracy=0.9919 ood_auroc=0.9216 sure_ds_f1=0.7317 conformal_cov=0.9919
[OOD] kanit=real_ood_split durum=failed gecti=False
[DONE] Dogrulama ve held-out test artefaktlari kaydedildi.
